# Emotion Recognition from Speech - Data Exploration

This notebook provides tools for exploring and visualizing audio data for emotion recognition.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import seaborn as sns
from IPython.display import Audio, display

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## 1. Load and Examine Audio Files

In [ ]:
# Configuration
DATA_DIR = '../data/'
SAMPLE_RATE = 22050
DURATION = 3.0  # seconds

# List available audio files
audio_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(('.wav', '.mp3', '.flac')):
            audio_files.append(os.path.join(root, file))

print(f"Found {len(audio_files)} audio files")
if audio_files:
    print(f"\nFirst 5 files:")
    for f in audio_files[:5]:
        print(f"  {f}")

## 2. Visualize Audio Waveforms

In [ ]:
def plot_waveform(audio_path, title="Waveform"):
    """Plot the waveform of an audio file."""
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
    
    plt.figure(figsize=(12, 4))
    librosa.display.waveshow(y, sr=sr, alpha=0.8)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.title(title)
    plt.tight_layout()
    plt.show()
    
    # Display audio player
    print(f"\nFile: {audio_path}")
    display(Audio(audio_path))

# Plot first audio file if available
if audio_files:
    plot_waveform(audio_files[0], "Sample Waveform")

## 3. Visualize Spectrograms

In [ ]:
def plot_spectrogram(audio_path, title="Spectrogram"):
    """Plot the spectrogram of an audio file."""
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
    
    # Compute mel spectrogram
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    plt.figure(figsize=(12, 6))
    librosa.display.specshow(
        mel_spec_db, 
        sr=sr, 
        x_axis='time', 
        y_axis='mel'
    )
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()

if audio_files:
    plot_spectrogram(audio_files[0], "Mel Spectrogram")

## 4. Extract and Visualize MFCCs

In [ ]:
def plot_mfcc(audio_path, title="MFCCs"):
    """Plot MFCCs of an audio file."""
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
    
    # Extract MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    
    plt.figure(figsize=(12, 6))
    librosa.display.specshow(
        mfccs, 
        sr=sr, 
        x_axis='time'
    )
    plt.colorbar()
    plt.title(title)
    plt.ylabel('MFCC Coefficients')
    plt.tight_layout()
    plt.show()
    
    print(f"MFCC shape: {mfccs.shape}")

if audio_files:
    plot_mfcc(audio_files[0], "MFCC Features")

## 5. Compare Features Across Emotions

In [ ]:
from src.utils import get_emotion_from_filename

# Group files by emotion
emotion_files = {}
for f in audio_files[:20]:  # Limit to first 20 for demo
    emotion = get_emotion_from_filename(os.path.basename(f))
    if emotion not in emotion_files:
        emotion_files[emotion] = []
    emotion_files[emotion].append(f)

print("Emotions found:")
for emotion, files in emotion_files.items():
    print(f"  {emotion}: {len(files)} files")

In [ ]:
def compare_mfccs(audio_files_dict, n_samples=3):
    """Compare MFCCs across different emotions."""
    n_emotions = min(len(audio_files_dict), 5)
    emotions = list(audio_files_dict.keys())[:n_emotions]
    
    fig, axes = plt.subplots(n_emotions, n_samples, figsize=(15, 3*n_emotions))
    
    for i, emotion in enumerate(emotions):
        files = audio_files_dict[emotion][:n_samples]
        for j, f in enumerate(files):
            y, sr = librosa.load(f, sr=SAMPLE_RATE)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
            
            ax = axes[i, j] if n_emotions > 1 else axes[j]
            librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax)
            ax.set_title(f"{emotion}" if j == 0 else "")
            ax.set_ylabel('MFCC' if j == 0 else '')
    
    plt.suptitle('MFCC Comparison Across Emotions', fontsize=14)
    plt.tight_layout()
    plt.show()

if emotion_files:
    compare_mfccs(emotion_files)

## 6. Audio Statistics

In [ ]:
def compute_audio_stats(audio_path):
    """Compute statistics for an audio file."""
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
    
    stats = {
        'duration': len(y) / sr,
        'rms_energy': np.sqrt(np.mean(y**2)),
        'zero_crossing_rate': np.mean(librosa.feature.zero_crossing_rate(y)),
        'spectral_centroid': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
        'spectral_bandwidth': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
    }
    
    return stats

# Compute stats for first few files
if audio_files:
    print("Audio Statistics:")
    print("=" * 60)
    for f in audio_files[:5]:
        stats = compute_audio_stats(f)
        print(f"\n{os.path.basename(f)}:")
        for key, value in stats.items():
            print(f"  {key:20s}: {value:.4f}")

## 7. Dataset Summary

Run this to get a complete overview of your dataset.

In [ ]:
def dataset_summary(data_dir):
    """Generate a summary of the dataset."""
    audio_files = []
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if file.endswith(('.wav', '.mp3', '.flac')):
                audio_files.append(os.path.join(root, file))
    
    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"\nTotal audio files: {len(audio_files)}")
    
    if audio_files:
        # Compute statistics
        durations = []
        for f in audio_files[:100]:  # Sample for speed
            y, sr = librosa.load(f, sr=SAMPLE_RATE, duration=10)
            durations.append(len(y) / sr)
        
        print(f"\nDuration statistics (sampled {min(100, len(audio_files))} files):")
        print(f"  Mean: {np.mean(durations):.2f}s")
        print(f"  Std:  {np.std(durations):.2f}s")
        print(f"  Min:  {np.min(durations):.2f}s")
        print(f"  Max:  {np.max(durations):.2f}s")
    
    print("\n" + "=" * 60)

dataset_summary(DATA_DIR)